In [7]:
import os
import pandas as pd

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        path = os.path.join(root, f)
        print(f"\n{path}")
        try:
            df = pd.read_csv(path)
            if df.shape[1] == 1:  # probably tab-separated, no header
                df = pd.read_csv(path, sep="\t", header=None)
            print(df.shape)
            display(df.head(3))
        except Exception as e:
            print("Could not read:", e)


/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/README
(32, 1)


,0
0,LIAR: A BENCHMARK DATASET FOR FAKE NEWS DETECTION
1,"William Yang Wang, ""Liar, Liar Pants on Fire"":..."
2,==============================================...



/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/test.tsv
Could not read: Error tokenizing data. C error: Expected 1 fields in line 3, saw 3


/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/train.tsv
Could not read: Error tokenizing data. C error: Expected 3 fields in line 8, saw 4


/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/valid.tsv
Could not read: Error tokenizing data. C error: Expected 5 fields in line 12, saw 6



In [6]:
import pandas as pd

cols = ["id", "label", "statement", "subject", "speaker", "job_title",
        "state", "party", "barely_true_ct", "false_ct", "half_true_ct",
        "mostly_true_ct", "pants_fire_ct", "context"]

base = "/kaggle/input/datasets/csmalarkodi/liar-fake-news-dataset/"
train = pd.read_csv(base + "train.tsv", sep="\t", header=None, names=cols)
valid = pd.read_csv(base + "valid.tsv", sep="\t", header=None, names=cols)
test  = pd.read_csv(base + "test.tsv",  sep="\t", header=None, names=cols)

print(train.shape, valid.shape, test.shape)
print(train["label"].value_counts())
train[["label", "statement"]].head(10)

(10240, 14) (1284, 14) (1267, 14)
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


,label,statement
0,false,Says the Annies List political group supports ...
1,half-true,When did the decline of coal start? It started...
2,mostly-true,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,Health care reform legislation is likely to ma...
4,half-true,The economic turnaround started at the end of ...
5,true,The Chicago Bears have had more starting quart...
6,barely-true,Jim Dunnam has not lived in the district he re...
7,half-true,I'm the only person on this stage who has work...
8,half-true,"However, it took $19.5 million in Oregon Lotte..."
9,mostly-true,Says GOP primary opponents Glenn Grothman and ...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

fake_labels = ["pants-fire", "false", "barely-true"]
for d in (train, valid, test):
    d["target"] = d["label"].isin(fake_labels).astype(int)

print(train["target"].value_counts(normalize=True))
print(train["statement"].str.split().str.len().describe())

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = vec.fit_transform(train["statement"])
X_val = vec.transform(valid["statement"])

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, train["target"])
pred = clf.predict(X_val)

print(classification_report(valid["target"], pred, target_names=["real", "misinfo"]))
print(confusion_matrix(valid["target"], pred))

target
0    0.561719
1    0.438281
Name: proportion, dtype: float64
count    10240.000000
mean        18.010059
std          9.658572
min          2.000000
25%         12.000000
50%         17.000000
75%         22.000000
max        467.000000
Name: statement, dtype: float64
              precision    recall  f1-score   support

        real       0.64      0.64      0.64       668
     misinfo       0.61      0.60      0.61       616

    accuracy                           0.62      1284
   macro avg       0.62      0.62      0.62      1284
weighted avg       0.62      0.62      0.62      1284

[[430 238]
 [244 372]]


In [9]:
from sklearn.metrics import f1_score

for mf in [5000, 20000, 50000]:
    for ng in [(1, 1), (1, 2)]:
        for C in [0.1, 1, 10]:
            v = TfidfVectorizer(max_features=mf, ngram_range=ng)
            Xt = v.fit_transform(train["statement"])
            m = LogisticRegression(C=C, max_iter=1000, class_weight="balanced")
            m.fit(Xt, train["target"])
            f1 = f1_score(valid["target"], m.predict(v.transform(valid["statement"])), average="macro")
            print(mf, ng, C, round(f1, 3))

5000 (1, 1) 0.1 0.623
5000 (1, 1) 1 0.618
5000 (1, 1) 10 0.574
5000 (1, 2) 0.1 0.624
5000 (1, 2) 1 0.618
5000 (1, 2) 10 0.591
20000 (1, 1) 0.1 0.626
20000 (1, 1) 1 0.615
20000 (1, 1) 10 0.595
20000 (1, 2) 0.1 0.624
20000 (1, 2) 1 0.624
20000 (1, 2) 10 0.598
50000 (1, 1) 0.1 0.626
50000 (1, 1) 1 0.615
50000 (1, 1) 10 0.595
50000 (1, 2) 0.1 0.622
50000 (1, 2) 1 0.627
50000 (1, 2) 10 0.616


In [10]:
pd.set_option("display.max_colwidth", None)

vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vec.fit_transform(train["statement"])
clf = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
clf.fit(X_train, train["target"])

valid["pred"] = clf.predict(vec.transform(valid["statement"]))
wrong = valid[valid["pred"] != valid["target"]]

print("Errors by original label:")
print(wrong["label"].value_counts())
print("\nError rate by original label:")
print((valid["pred"] != valid["target"]).groupby(valid["label"]).mean().round(2))

wrong[["label", "pred", "statement"]].sample(10, random_state=1)

Errors by original label:
label
barely-true    113
half-true      109
false          102
mostly-true     86
true            40
pants-fire      28
Name: count, dtype: int64

Error rate by original label:
label
barely-true    0.48
false          0.39
half-true      0.44
mostly-true    0.34
pants-fire     0.24
true           0.24
dtype: float64


,label,pred,statement
796,barely-true,0,Says Scott Walkers jobs agency gives financial incentives that lead Wisconsin companies to look at shipping jobs elsewhere. ... A company got $4.6 million in taxpayer money and actually reduced its jobs by 300 people.
782,barely-true,0,"A year since the Recovery Summer, there has been been a fall, winter and spring of no recovery."
98,pants-fire,0,The results of Atlanta Public Schools 2010 and 2011 CRCT tests have not been questioned.
1033,half-true,1,The accuracy of the Obama tax calculator
1150,barely-true,0,The Fed created $15 trillion in the bailout process and $5 trillion went overseas.
992,barely-true,0,Weve used (generally accepted accounting principles) here in this building the last two years.
953,true,1,Obama has admitted a cap and trade plan would cause electricity bills to skyrocket.
198,false,0,"Several times, Obama has apologized for what he deems to be American arrogance, dismissiveness, and derision; for dictating solutions, for acting unilaterally ... ."
686,mostly-true,1,One third of our age group (Millennials) have moved back in with their parents.
85,barely-true,0,"Toledo is fourth in the nation behind much bigger cities Miami, Las Vegas, Portland in number of human trafficking arrests and rescues."


We loaded the LIAR dataset. Looked at the labels and they were 6, so we reduced that down to 2 as we wanted our classifier to output binary (real and misinfo). Trained a very simple model to get a baseline, a basic idea of where we are at, so that we can go on and build a better one later on. 

We've tried various settings in an attempt to get better results with the TF-IDF vectorisation. Every combination gave us a result between 0.57 and 0.63, with nearly all of them in the 0.60-0.63 range. So tuning makes no difference at all almost and the model has clearly hit its limit.

So for the average case, let's say we have 62% accuracy. We now have to consider what the errors are in the remaining 38%. Some of the true statements are being written in an emotional way which makes them appear to be false. Some of the false statements are being written in an extremely flat, deadpan way which makes them appear to be true. 

we can rectify the errors in the following ways:
1. the model we have used only reads words, not meaning, so we are trying to shift to BERT
2. this model can't check facts, so further we'll introduce a search factor

we'll get to know more ways to fix as we go further